# Lab 1

Load data from the provided csv file and print .head() to verify correctness of data

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")

print(df.head())
print(df.info())


## Pre analisys 

Some of the columns had strange names so i slightly renamed them

In [ ]:
# Questionalble columns 
# I want to rename some columns to make them more descriptive
df.rename(columns={
    "cust_id": "customer_id",
    "bi_st": "billing_status",
    "ref_num": "reference_number",
    "sku": "product_variant_sku",
}, inplace=True)

# Change types of some columns to more appropriate ones
df = df.astype({
    "qty_ordered": "int64",
    "customer_id": "int64",
    "item_id": "int64"
})

df.to_csv("data_cleaned.csv", index=False)

df.info()

## Analysis

1) Which month had the highest average order total?


(Computes average order `total` per month to find months with the largest average order value.)

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
# per-order total in case multiple rows per order_id
if 'order_id' in df.columns:
    orders = df.groupby('order_id').agg({'total':'sum','order_date':'first'}).reset_index()
    monthly_avg = orders.groupby(orders['order_date'].dt.to_period('M'))['total'].mean()
else:
    # fallback: average line-item totals by month
    monthly_avg = df.groupby(df['order_date'].dt.to_period('M'))['total'].mean()
print('Monthly average order total (top 5):', monthly_avg.sort_values(ascending=False).head())
print('Month with highest average order total:', monthly_avg.idxmax(), monthly_avg.max())

2) Which day of week has the highest average order total?

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
dow_avg = df.groupby(df['order_date'].dt.day_name())['total'].mean().sort_values(ascending=False)
print('Average total by day of week:', dow_avg)
print('Top day:', dow_avg.idxmax(), dow_avg.max())

3) Top 10 SKUs by total revenue

In [ ]:
top_skus = df.groupby('product_variant_sku')['total'].sum().sort_values(ascending=False).head(10)
print('Top 10 SKUs by revenue:', top_skus)

4) Which Region had the highest average discount percent?

In [ ]:
if 'Discount_Percent' in df.columns and 'Region' in df.columns:
    region_disc = df.groupby('Region')['Discount_Percent'].mean().sort_values(ascending=False)
    print('Average discount percent by Region:', region_disc)
    print('Top region:', region_disc.idxmax(), region_disc.max())
else:
    print('Required columns not present: Discount_Percent and/or Region')

5) What percentage of orders (rows) were canceled?

In [ ]:
if 'status' in df.columns:
    pct_canceled = (df['status'] == 'canceled').mean() * 100
    print(f"Canceled rows: {pct_canceled:.2f}%")
    print(df['status'].value_counts().head())
else:
    print('Column `status` not found in df')

6) Top 10 customers by total spend

In [ ]:
cust_col = 'customer_id' if 'customer_id' in df.columns else 'cust_id' if 'cust_id' in df.columns else None
if cust_col is not None:
    top_customers = df.groupby(cust_col)['total'].sum().sort_values(ascending=False).head(10)
    print('Top customers by total spend:', top_customers)
else:
    print('No customer id column found')

7) Average order value (AOV) by billing type (Net/Gross) or `billing_status`

In [ ]:
billing_col = 'billing_status' if 'billing_status' in df.columns else 'bi_st' if 'bi_st' in df.columns else None
if billing_col is not None:
    aov_by_billing = df.groupby(billing_col)['total'].mean().round(2).sort_values(ascending=False)
    print('AOV by billing type:', aov_by_billing)
else:
    print('No billing column found (bi_st or billing_status)')

8) How many unique customers are there?

In [ ]:
cust_col = 'customer_id' if 'customer_id' in df.columns else 'cust_id' if 'cust_id' in df.columns else None
if cust_col is not None:
    print('Unique customers:', df[cust_col].nunique())
else:
    print('No customer id column found')

9) What is the most common order status?

In [ ]:
if 'status' in df.columns:
    vc = df['status'].value_counts()
    print('Top statuses:', vc.head())
    print('Most common:', vc.idxmax(), vc.max())
else:
    print('Column `status` not found')

10) Average items per order (average total quantity per `order_id`)

In [ ]:
if 'order_id' in df.columns and 'qty_ordered' in df.columns:
    qty_per_order = df.groupby('order_id')['qty_ordered'].sum()
    print('Average items per order:', qty_per_order.mean().round(2))
    print('Median items per order:', qty_per_order.median())
else:
    print('Required columns `order_id` or `qty_ordered` missing')

11) What fraction of rows had a discount applied?

In [ ]:
if 'discount_amount' in df.columns:
    frac = (df['discount_amount'] > 0).mean() * 100
    print(f'Rows with discount: {frac:.2f}%')
else:
    print('Column `discount_amount` not present')

12) Correlation between discount amount and order total (are bigger discounts associated with larger totals?)

In [ ]:
cols = ['discount_amount','total']
if all(c in df.columns for c in cols):
    corr = df['discount_amount'].corr(df['total'])
    print('Correlation (discount_amount vs total):', round(corr, 4))
    print('Interpretation: near 0 => weak linear relation; positive/negative show direction')
else:
    print('Required columns missing:', [c for c in cols if c not in df.columns])